In [53]:
import pandas as pd
import numpy as np
pd.options.display.float_format = '{:,.2f}'.format

In [37]:
df26_clean = pd.read_parquet('df26_clean.parquet')
df26_clean = df26_clean.groupby(['группа_ка', 'сегмент_ка', 'контрагент'], as_index=False)['оборот, тыс.ед.'].sum()
df26_clean.head()

,группа_ка,сегмент_ка,контрагент,"оборот, тыс.ед."
0,3 лица,3 лица,Арцибеев Алексей Валерьевич,8.63
1,3 лица,3 лица,Безруков Константин Львович,1.00
2,3 лица,3 лица,ВМБ-Сервис АО,725.76
3,3 лица,3 лица,ВЫМПЕЛКОМ ПАО,191.66
4,3 лица,3 лица,Зиброва Наталья Михайловна,8.47


In [45]:
df_9008 = pd.read_parquet('df_9008.parquet')
df_9008

,ном_группа,"оборот, тыс.ед.",сегмент
0,Горох Урожай 2024,"24,345.19",Растениеводство
1,Горох Урожай 2025,"50,563.44",Растениеводство
2,Горчица Урожай 2024,"16,457.80",Растениеводство
3,Горчица Урожай 2025,445.71,Растениеводство
4,Естественные травы Урожай 2025,8.58,Растениеводство
5,Изготовление запчастей,429.23,Растениеводство
6,Комбикорм для овец Рост,363.88,МРС
7,Комбикорм для овец Старт,13.46,МРС
8,Комбикорм для овец Финиш,"1,042.93",МРС
9,Кукуруза Урожай 2024,"8,925.04",Растениеводство


In [43]:
df26_clean

,группа_ка,сегмент_ка,контрагент,"оборот, тыс.ед."
0,3 лица,3 лица,Арцибеев Алексей Валерьевич,8.63
1,3 лица,3 лица,Безруков Константин Львович,1.00
2,3 лица,3 лица,ВМБ-Сервис АО,725.76
3,3 лица,3 лица,ВЫМПЕЛКОМ ПАО,191.66
4,3 лица,3 лица,Зиброва Наталья Михайловна,8.47
5,3 лица,3 лица,ИНФО.АУДИТ ООО,420.00
6,3 лица,3 лица,Интерфакс АО,8.08
7,3 лица,3 лица,КИТО ООО,"38,448.45"
8,3 лица,3 лица,Каменцев Владимир Николаевич,6.50
9,3 лица,3 лица,Коньков Роман Александрович,7.90


In [47]:
df26_clean.to_excel('df26_clean.xlsx')
df_9008.to_excel('df_9008.xlsx')

In [61]:
# 1. Расчёт долей ном_групп в общих расходах
total_9008 = df_9008['оборот, тыс.ед.'].sum()
df_9008['доля_ном_группы'] = df_9008['оборот, тыс.ед.'] / total_9008

# 2. Cross-join: каждая строка df26_clean × каждая ном_группа
df_cross = df26_clean.assign(key=1).merge(
    df_9008[['ном_группа', 'сегмент', 'доля_ном_группы']].assign(key=1), 
    on='key'
).drop('key', axis=1)

# 3. Распределение оборота пропорционально долям ном_групп
df_cross['оборот_распределенный'] = df_cross['оборот, тыс.ед.'] * df_cross['доля_ном_группы']

# 4. Определение вид_связи (векторизованно через np.select)
conditions = [
    df_cross['группа_ка'] == '3 лица',
    df_cross['группа_ка'] == 'Прочие ГАП',
    (df_cross['группа_ка'] == 'ГСК') & (df_cross['сегмент_ка'] == df_cross['сегмент']),
    (df_cross['группа_ка'] == 'ГСК') & (df_cross['сегмент_ка'] != df_cross['сегмент']),
]
choices = ['3 лица', 'Прочие ГАП', 'ГСК', 'межсегмент']
df_cross['вид_связи'] = np.select(conditions, choices, default='не_указано')

# 5. Добавление остатка (расходы без контрагентов: зарплата, материалы и т.д.)
total_26_clean = df26_clean['оборот, тыс.ед.'].sum()
remainder = total_9008 - total_26_clean

if remainder > 0:
    df_remainder = df_9008[['ном_группа', 'сегмент', 'доля_ном_группы']].copy()
    df_remainder['контрагент'] = 'Прочие расходы'
    df_remainder['группа_ка'] = '3 лица'
    df_remainder['сегмент_ка'] = '3 лица'
    df_remainder['вид_связи'] = '3 лица'
    df_remainder['оборот_распределенный'] = remainder * df_remainder['доля_ном_группы']
    
    df_result = pd.concat([df_cross, df_remainder], ignore_index=True)
else:
    df_result = df_cross

# 6. Финальная очистка и переименование
df_result = df_result.drop(columns=['оборот, тыс.ед.', 'доля_ном_группы'])
df_result = df_result.rename(columns={'оборот_распределенный': 'оборот, тыс.ед.'})

# Приведение типов к string
text_cols = ['ном_группа', 'сегмент', 'контрагент', 'группа_ка', 'сегмент_ка', 'вид_связи']
for col in text_cols:
    if col in df_result.columns:
        df_result[col] = df_result[col].astype('string')

In [63]:
df_result.to_excel('df98_result.xlsx')

In [9]:
df_main = pd.read_parquet('df_final.parquet')
df_main

,контрагент,ном_группа,счет,"оборот, тыс.ед.",доход_расход,вид_дохода_расхода,сегмент,группа_ка,сегмент_ка,вид_связи
0,АГРО-ЭКСПО ООО,Горох Урожай 2025,90.01,"-225,340.91",Выручка,Выручка,Растениеводство,3 лица,3 лица,3 лица
1,АГРО-ЭКСПО ООО,Пшеница озимая Урожай 2025,90.01,"-141,872.73",Выручка,Выручка,Растениеводство,3 лица,3 лица,3 лица
2,АГРОКУЛЬТУРА ООО,Прочая реализация ТМЦ,90.01,"-2,179.56",Выручка,Прочая операционная деятельность,Растениеводство,3 лица,3 лица,3 лица
3,АГРОТЭКС ООО,Прочая реализация ТМЦ,90.01,"-1,199.95",Выручка,Прочая операционная деятельность,Растениеводство,3 лица,3 лица,3 лица
4,АДМИНИСТРАЦИЯ МАНЬКОВО-БЕРЕЗОВСКОГО СЕЛЬСКОГО ...,УСЛУГИ ТРАКТОРНОГО ПАРКА ВНЕШНИЕ,90.01,-12.10,Выручка,Прочая операционная деятельность,Растениеводство,3 лица,3 лица,3 лица
...,...,...,...,...,...,...,...,...,...,...
890,не_указано,Подсолнечник Урожай 2025,90.02,"123,890.05",Изменение в оценке,Посевы изменение в оценке,не_указано,не_указано,не_указано,не_указано
891,не_указано,Пшеница озимая Урожай 2024,90.02,"-23,136.16",Изменение в оценке,Посевы изменение в оценке,не_указано,не_указано,не_указано,не_указано
892,не_указано,Пшеница озимая Урожай 2025,90.02,"158,234.71",Изменение в оценке,Посевы изменение в оценке,не_указано,не_указано,не_указано,не_указано
893,не_указано,Ячмень Урожай 2024,90.02,"-8,412.17",Изменение в оценке,Посевы изменение в оценке,не_указано,не_указано,не_указано,не_указано


In [65]:
df_oll = pd.read_parquet('intermediate_data.parquet')

In [69]:
df_oll.loc[df_oll['Дт'].str.startswith('90.08'), 'Сумма'].sum()

np.float64(783222882.12)